# `orders` — profile, choose columns & trim

Trim type: **row**. Choose **7 columns**, **1,200,000 rows**. Save `orders_7c_1200000r.csv`.

> Output columns: `c0`…`c6`.

In [3]:
import os
import numpy as np
import pandas as pd

NAME       = "orders"
N_ROWS     = 1500000
N_COLS     = 7
TRIM_ROWS  = "head"   # head | sample

RAW_PATH   = "orders.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ";"
OUT_DELIM  = DELIM
HAS_HEADER = False
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [2]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (1500000, 9)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8']


,c0,c1,c2,c3,c4,c5,c6,c7,c8
0,1,36901,O,173665.47,1996-01-02,5-LOW,Clerk#000000951,0,nstructions sleep furiously among
1,2,78002,O,46929.18,1996-12-01,1-URGENT,Clerk#000000880,0,"foxes. pending accounts at the pending, silen..."
2,3,123314,F,193846.25,1993-10-14,5-LOW,Clerk#000000955,0,sly final accounts boost. carefully regular id...
3,4,136777,O,32151.78,1995-10-11,5-LOW,Clerk#000000124,0,"sits. slyly regular warthogs cajole. regular, ..."
4,5,44485,F,144659.20,1994-07-30,5-LOW,Clerk#000000925,0,quickly. bold deposits sleep slyly. packages u...


In [ ]:
raw.dtypes

## 2. Profile: cardinality, top-value %, and group skew

In [4]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,1500000,0,0.0,100.00,0.00,1.00,1,1.00
c1,99996,0,0.0,6.67,0.00,15.00,41,2.73
c2,3,0,0.0,0.00,48.80,500000.00,732044,1.46
c3,1464556,0,0.0,97.64,0.00,1.02,4,3.91
c4,2406,0,0.0,0.16,0.05,623.44,702,1.13
c5,5,0,0.0,0.00,20.04,300000.00,300589,1.00
c6,1000,0,0.0,0.07,0.11,1500.00,1618,1.08
c7,1,0,0.0,0.00,100.00,1500000.00,1500000,1.00
c8,1482071,0,0.0,98.80,0.00,1.01,17,16.80


## 3. Choose columns (cardinality mix + id/super-key)

In [5]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6']


## 4. Trim to the chosen columns x exact rows

In [6]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (1500000, 7)


,c0,c1,c2,c3,c4,c5,c6
0,1,36901,O,173665.47,1996-01-02,5-LOW,Clerk#000000951
1,2,78002,O,46929.18,1996-12-01,1-URGENT,Clerk#000000880
2,3,123314,F,193846.25,1993-10-14,5-LOW,Clerk#000000955
3,4,136777,O,32151.78,1995-10-11,5-LOW,Clerk#000000124
4,5,44485,F,144659.20,1994-07-30,5-LOW,Clerk#000000925


## 5. Save the trimmed CSV

In [7]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote orders_7c_1500000r.csv (1500000, 7)
reloaded: (1500000, 7)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6']


## 6. Check selected cardinality and skew

In [ ]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

In [ ]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof